
- Explain the complete processing pipeline from raw radar bytes to final CSV datasets.
- Show exactly how range-doppler and point-cloud data are parsed.
- Explain the final physical meaning of each output data field.

## 1) System Context and Data Flow

Source modules:
- Parsing_script/main.py
- Parsing_script/radar_application.py
- Parsing_script/payload.py
- Parsing_script/ui.py

High-level flow:
1. UI configures radar and opens command/data serial ports.
2. read_data() continuously reads bytes from data UART.
3. Byte stream is framed using TI magic word + packet length.
4. process_data() decodes frame header and TLV payloads.
5. Parsed structures are transformed into features and written to CSV.
6. Final data products are used for activity/social behavior analysis.

## 2) Runtime Architecture

In main.py:
- MainWindow is created for UI and plot/table updates.
- RadarApplication object is created for acquisition and parsing.
- Two worker threads run in parallel:
  - read_data(window): byte acquisition and frame extraction.
  - process_data(window): queued frame parsing and data handling.
- A periodic timer updates visual plots.


## 3) Frame Extraction from Raw Byte Stream

The parser reads a continuous byte stream from the data port and reconstructs complete radar frames.

Frame synchronization method:
- Search for magic word: 02 01 04 03 06 05 08 07.
- After alignment, read the fixed 40-byte frame header.
- Extract total packet length from header bytes [12:16] (little-endian).
- Wait until exactly that many bytes are available.
- Slice the packet as one complete frame and push it into the queue.

Robustness controls in code:
- Discard preamble garbage before magic word.
- Protect against oversized buffer growth.
- Skip frame enqueue when queue is full (with warning logs).

## 4) Frame Structure and TLV Parsing

A frame contains:
1. Header (40 bytes)
2. Payload made of multiple TLVs

Header fields (examples):
- Frame Number
- Num TLVs
- Total Packet Length

TLV parsing loop:
- Read TLV type (4 bytes) + TLV length (4 bytes)
- Slice TLV payload
- Route payload to parser function by TLV type

Important TLVs used for final datasets:
- Type 1: Detected points (point cloud)
- Type 5: Range-Doppler heatmap
- Type 7: Side info (SNR/noise)

## 5) How Point Cloud is Parsed (TLV Type 1)

### 5.1 TLV payload structure
The parser in payload.py treats Type 1 payload as a repeated fixed record.
- record size = 16 bytes
- byte [0:4]   -> x (float32, little-endian)
- byte [4:8]   -> y (float32, little-endian)
- byte [8:12]  -> z (float32, little-endian)
- byte [12:16] -> doppler (float32, little-endian)

Number of parsed points is computed as:
- num_detected_obj = len(payload) / 16

### 5.2 Parsing algorithm in code
For each point index i:
1. start_index = i * 16
2. unpack x, y, z, doppler using struct.unpack('<f', ...)
3. create dictionary: { 'X': x, 'Y': y, 'Z': z, 'Doppler': doppler }
4. append to detected_points list

Error handling:
- If a 4-byte float cannot be unpacked (struct.error), parser logs error and skips only that point.

### 5.3 Feature engineering before persistence
In radar_application.py (handle_data), each parsed point is transformed before CSV write:
- range = sqrt(x*x + y*y + z*z)
- aoa = degrees(arctan2(y, x))

Why arctan2 is used:
- arctan2 preserves quadrant information and avoids divide-by-zero issues when x is near zero.

### 5.4 Final CSV row schema
Each row written to points_cloud.csv:
- frame_number
- point_index
- x, y, z
- doppler
- range
- aoa

Physical meaning:
- x, y, z: 3D reflector location in radar coordinate frame
- doppler: radial velocity estimate for that reflector
- range: Euclidean distance from radar
- aoa: horizontal arrival angle of reflection

## 6) How Range-Doppler is Parsed (TLV Type 5)

### 6.1 Matrix dimensions used by parser
Type 5 parser in payload.py first resolves matrix size from radar_params:
- range_fft_size = Number of Samples per Chirp
- doppler_fft_size:
  - Number of Chirps in Frame when waveform multiplexing is Phased-Array
  - otherwise Number of Chirps in Loop

Expected payload size check:
- expected_length = range_fft_size * doppler_fft_size * 2 bytes
- If payload length differs, parser raises ValueError for that TLV.

### 6.2 Byte-to-matrix conversion
Each heatmap cell is parsed as an unsigned 16-bit value from 2 bytes (little-endian).
Nested loop structure:
1. for each range bin r in [0, range_fft_size)
2. for each doppler bin d in [0, doppler_fft_size)
3. read bytes at index (r * doppler_fft_size + d) * 2
4. convert to integer and append into row
5. append row into heatmap

Output from parser:
- 2D list heatmap[r][d] containing signal strength values.

### 6.3 Processing before visualization and storage
In update_plots_wrapper:
- heatmap is fft-shifted along doppler axis for centered velocity visualization
- then interpolated/rescaled for display resolution

In handle_data CSV path:
- no interpolation is written to CSV
- raw parsed bin values are flattened into long table rows

### 6.4 Final CSV row schema
Each row written to range_doppler.csv:
- timestamp_s
- frame_number
- range_bin
- doppler_bin
- signal_strength

Physical meaning:
- range_bin: distance-axis discrete index
- doppler_bin: radial-speed-axis discrete index
- signal_strength: reflection intensity/energy at that bin

## 7) How Side Information is Parsed (TLV Type 7)

### 7.1 TLV payload structure
Type 7 provides per-detected-point quality metrics.
Each point uses 4 bytes:
- bytes [0:2] -> raw SNR (uint16, little-endian)
- bytes [2:4] -> raw noise (uint16, little-endian)

Number of points in side-info payload:
- num_points = len(payload) // 4

### 7.2 Parsing algorithm
For each point index i:
1. extract snr_bytes = payload[i*4 : i*4+2]
2. extract noise_bytes = payload[i*4+2 : i*4+4]
3. convert both using int.from_bytes(..., little-endian)
4. scale to dB using 0.1 factor
5. append dictionary: { 'snr': snr_db, 'noise': noise_db }

Scaling rule implemented in payload.py:
- snr_db = raw_snr * 0.1
- noise_db = raw_noise * 0.1

### 7.3 How it is linked to frame context
In handle_data:
- side-info is associated with current frame_number
- point_index is used to keep correspondence with detected points list
- each point gets one CSV row

Final CSV schema in noise_snr.csv:
- timestamp_s
- frame_number
- point_index
- snr
- noise

Physical meaning:
- snr: confidence proxy for that detection (higher usually more reliable)
- noise: local noise floor estimate near that detection

## 8) Final Output Data Products

### points_cloud.csv
Columns:
- frame_number
- point_index
- x, y, z
- doppler
- range
- aoa

Physical meaning:
- 3D location and radial motion of each detected reflector point.
- range and aoa provide geometry-centric features for activity analysis.

### range_doppler.csv
Columns:
- timestamp_s
- frame_number
- range_bin
- doppler_bin
- signal_strength

Physical meaning:
- Spatiokinematic energy map of the scene in distance-velocity domain.
- Useful for motion pattern signatures over time.

### noise_snr.csv
Columns:
- timestamp_s
- frame_number
- point_index
- snr
- noise

Physical meaning:
- Per-point detection quality and local noise context.
- Supports confidence weighting and data quality filtering.

## 9) End-to-End Step-by-Step Process Summary

1. Configure radar and connect serial ports (command/data).
2. Send configuration to radar from UI.
3. Continuously read data UART byte stream.
4. Detect frame boundaries using magic word.
5. Extract full frame using packet length.
6. Parse frame header to get frame metadata and TLV count.
7. Parse TLV payloads by type (points, range-doppler, side-info, etc.).
8. Convert parsed structures into final per-row CSV format.
9. Compute derived geometric features (range, aoa).
10. Persist outputs as points_cloud.csv, range_doppler.csv, noise_snr.csv.